In [56]:
from typing import Annotated, Literal, Sequence, TypedDict
import langchainhub as hub
from langchain_core.messages import BaseMessage, HumanMessage
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import PromptTemplate
from pydantic import BaseModel, Field
from langgraph.graph.message import add_messages
from langgraph.prebuilt import tools_condition
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.vectorstores import Chroma
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.tools import create_retriever_tool
from langgraph.graph import END, StateGraph, START
from langgraph.prebuilt import ToolNode

In [57]:
import langchain
import importlib.metadata

print("LangChain Version:", langchain.__version__)
print("LangGraph Version:", importlib.metadata.version("langgraph"))

LangChain Version: 1.3.0
LangGraph Version: 1.2.0


In [58]:
from langgraph.graph import StateGraph


In [59]:
!pip install sentence-transformers


In [60]:
from langchain_huggingface import HuggingFaceEmbeddings
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-V2")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4721.97it/s]


In [61]:
from langchain_groq import ChatGroq

In [62]:
# then needed embedding model and llm
# making object for the model
llm=ChatGroq(model_name="llama-3.3-70b-versatile")

In [63]:
llm.invoke("hi hello world")

AIMessage(content="Hello. It's nice to meet you. Is there something I can help you with or would you like to chat?", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 25, 'prompt_tokens': 38, 'total_tokens': 63, 'completion_time': 0.051974671, 'completion_tokens_details': None, 'prompt_time': 0.005482566, 'prompt_tokens_details': None, 'queue_time': 0.05636507, 'total_time': 0.057457237}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_dae98b5ecb', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019e2cd2-2c1f-7e31-a9e8-9f5773b60396-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 38, 'output_tokens': 25, 'total_tokens': 63})

In [ ]:
from langchain_community.document_loaders import PyMuPDFLoader

def load_pdf(filepath):

    loader = PyMuPDFLoader(filepath)

    documents = loader.load()

    # optional cleanup
    for doc in documents:
        doc.page_content = (
            doc.page_content
            .replace("\n", " ")
            .replace("\t", " ")
            .strip()
        )

    return documents # loading pdf file path

In [77]:
# to see first 500 char 
pdf_text = load_pdf("Avneet_Kaur_Research_Document.pdf")
print(pdf_text[:500])   # preview first 500 characters

[Document(metadata={'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-05-15T18:22:15+00:00', 'source': 'Avneet_Kaur_Research_Document.pdf', 'file_path': 'Avneet_Kaur_Research_Document.pdf', 'total_pages': 22, 'format': 'PDF 1.4', 'title': 'Avneet Kaur – A Comprehensive Research Document', 'author': 'Research Desk', 'subject': 'Indian Entertainment Industry Profile', 'keywords': '', 'moddate': '2026-05-15T18:22:15+00:00', 'trapped': '', 'modDate': "D:20260515182215+00'00'", 'creationDate': "D:20260515182215+00'00'", 'page': 0}, page_content='AVNEET KAUR  |  A Comprehensive Research Document © Research Desk  2026 Page 1 AVNEET KAUR A Comprehensive Research Document From Dance Reality Shows to International Cinema Actress · Dancer · Social Media Influencer Career Span: 2010 – Present | Base: Mumbai, India Prepared: May 2026 15-Page Research Overview | Audience: General Readers'), Document(metadata={'producer': 'ReportLab PDF Library - (o

In [ ]:
pdf_text # to see innside the  pdf

[Document(metadata={'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-05-15T18:22:15+00:00', 'source': 'Avneet_Kaur_Research_Document.pdf', 'file_path': 'Avneet_Kaur_Research_Document.pdf', 'total_pages': 22, 'format': 'PDF 1.4', 'title': 'Avneet Kaur – A Comprehensive Research Document', 'author': 'Research Desk', 'subject': 'Indian Entertainment Industry Profile', 'keywords': '', 'moddate': '2026-05-15T18:22:15+00:00', 'trapped': '', 'modDate': "D:20260515182215+00'00'", 'creationDate': "D:20260515182215+00'00'", 'page': 0}, page_content='AVNEET KAUR  |  A Comprehensive Research Document © Research Desk  2026 Page 1 AVNEET KAUR A Comprehensive Research Document From Dance Reality Shows to International Cinema Actress · Dancer · Social Media Influencer Career Span: 2010 – Present | Base: Mumbai, India Prepared: May 2026 15-Page Research Overview | Audience: General Readers'),
 Document(metadata={'producer': 'ReportLab PDF Library - (

In [ ]:
pdf_text[0].metadata # meta data data about the data ,, loading

{'producer': 'ReportLab PDF Library - (opensource)',
 'creator': '(unspecified)',
 'creationdate': '2026-05-15T18:22:15+00:00',
 'source': 'Avneet_Kaur_Research_Document.pdf',
 'file_path': 'Avneet_Kaur_Research_Document.pdf',
 'total_pages': 22,
 'format': 'PDF 1.4',
 'title': 'Avneet Kaur – A Comprehensive Research Document',
 'author': 'Research Desk',
 'subject': 'Indian Entertainment Industry Profile',
 'keywords': '',
 'moddate': '2026-05-15T18:22:15+00:00',
 'trapped': '',
 'modDate': "D:20260515182215+00'00'",
 'creationDate': "D:20260515182215+00'00'",
 'page': 0}

In [ ]:
#  chunking
# split text into chunks

def split_text(documents):
    splitter = RecursiveCharacterTextSplitter(
        chunk_size = 500,  # size of each chunk 500 
        chunk_overlap = 50 # consecutive chunks share 50 characters to preserve context at boundaries
    )
    return splitter.split_documents(documents) # Returns a list of Document objects


In [ ]:
# split documents 
docs_split = split_text(pdf_text) # splitted after cunks
docs_split# down chunks represented 

[Document(metadata={'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-05-15T18:22:15+00:00', 'source': 'Avneet_Kaur_Research_Document.pdf', 'file_path': 'Avneet_Kaur_Research_Document.pdf', 'total_pages': 22, 'format': 'PDF 1.4', 'title': 'Avneet Kaur – A Comprehensive Research Document', 'author': 'Research Desk', 'subject': 'Indian Entertainment Industry Profile', 'keywords': '', 'moddate': '2026-05-15T18:22:15+00:00', 'trapped': '', 'modDate': "D:20260515182215+00'00'", 'creationDate': "D:20260515182215+00'00'", 'page': 0}, page_content='AVNEET KAUR  |  A Comprehensive Research Document © Research Desk  2026 Page 1 AVNEET KAUR A Comprehensive Research Document From Dance Reality Shows to International Cinema Actress · Dancer · Social Media Influencer Career Span: 2010 – Present | Base: Mumbai, India Prepared: May 2026 15-Page Research Overview | Audience: General Readers'),
 Document(metadata={'producer': 'ReportLab PDF Library - (

In [94]:
vector_store = Chroma.from_documents(
    documents=docs_split, # store
    colllection = "rag-chrome",
    embedding=embeddings
)

TypeError: Chroma.__init__() got an unexpected keyword argument 'colllection'

In [64]:
def AI_Assistant(state): # llm
    pass


In [65]:
def retriever(state): # retrive for relevant documents
    pass

In [66]:
def rewrite(state): # if document is not matching
    pass 

In [67]:
def generate(state):
    pass

In [68]:
class AgentState:
    pass

In [73]:
workflow = StateGraph(AgentState)
workflow.add_node("ai_assistant",AI_Assistant)
workflow.add_node("retriever",retriever)
workflow.add_node("rewrite",rewrite)
workflow.add_node("generate",generate) 
# node which helps in function representation


In [74]:
# edges for connectivity
workflow.add_edge(START, "ai_assistant")

workflow.add_edge("ai_assistant", "retriever")

workflow.add_edge("retriever", "generate")

workflow.add_edge("generate", END)# we need to start and end edges

In [ ]:
app = workflow.compile()
app.invoke("")